In [1]:
import pandas as pd
import numpy as np

In [2]:
# Chapter 10. Data Aggregation and Group Operations

In [3]:
# pandas provides a versatile .groupby() interface, enabling to slice and summarize datasets in a natural way
# query languages impose certain limitations on the kinds of group operations
# with the Python expressiveness can express as custom Python functions 

In [4]:
# 10.1 How to think About Group Operations

In [5]:
# group operations : split-apply-combine
# Series, DataFrame is split into groups based on one or more keys that you provide
# splitting is performed on a particular axis of an object
# once split, function is applied to each group, producing a new value
# results of all those function applications are combined into a result object

In [6]:
# each grouping key can take many forms, and the keys do not have to be all the same type:
# - A list or array of values that is the same length as the axis being grouped
# - A value indicating a column name in a DataFrame
# - A dictionary or Series giving a correspondence between the values on the axis being grouped and the group names
# - A function to be invoked on the axis index or the individuals labels in the index

In [7]:
df = pd.DataFrame({"key1":["a","a",None,"b","b","a",None],
                  "key2":pd.Series([1,2,1,2,1,None,1],dtype="Int64"),
                  "data1":np.random.standard_normal(7),
                  "data2":np.random.standard_normal(7)})
df

,key1,key2,data1,data2
0,a,1,-0.264609,-1.679621
1,a,2,-1.465616,-0.379823
2,None,1,-0.927017,0.051805
3,b,2,0.910231,0.362820
4,b,1,0.171047,-0.518360
5,a,<NA>,-0.922581,-0.976857
6,None,1,0.699284,-1.292389


In [8]:
# suppose you want to compute the mean of the data1 column using the labels from key1

In [9]:
# method 1 : access data1 and call groupby with the column (a Series) at key1:

grouped = df["data1"].groupby(df["key1"])
grouped

In [10]:
# grouped is now a special "GroupBy" object
# to compute group means we can call the GroupBy's .mean() method:
grouped.mean()

# here the data (a Series) has been aggregated by splitting the data on the group key;
# producing a new Series that is now indexed by the unique values in the key1 column
# the result index has the name "key1" because the DataFrame column df["key1"] did

key1
a   -0.884269
b    0.540639
Name: data1, dtype: float64

In [11]:
# if instead passed multiple arrays as a list, we'd get something different:
means = df["data1"].groupby([df["key1"],df["key2"]]).mean()
means

key1  key2
a     1      -0.264609
      2      -1.465616
b     1       0.171047
      2       0.910231
Name: data1, dtype: float64

In [12]:
# here we grouped the data using two keys, and the resulting Series now has a heirarchical index;
# which consist of the unique pairs of keys observed:
means.unstack()

key2,1,2
key1,,
a,-0.264609,-1.465616
b,0.171047,0.910231


In [13]:
# in this example, the group keys are all Series, though they could be any arrays of the right lenght:
states = np.array(["OH","CA","CA","OH","OH","CA","OH"])
years = [2005,2005,2006,2005,2006,2005,2006]
df["data1"].groupby([states,years]).mean()

CA  2005   -1.194099
    2006   -0.927017
OH  2005    0.322811
    2006    0.435165
Name: data1, dtype: float64

In [17]:
# frequently, the grouping information is the same DataFrame as the data you want to work on
# in this caes, you can pass column names as the group keys:
df.groupby("key1").mean()

,key2,data1,data2
key1,,,
a,1.5,-0.884269,-1.01210
b,1.5,0.540639,-0.07777


In [19]:
df.groupby("key2").mean()

# there is no key1 column in the result since df["key1"] is not numeric data
# non-numeric data is said to be a nuisance column which is automatically excluded from result
# by default, all of the numeric columns are aggregated

TypeError: agg function failed [how->mean,dtype->object]

In [20]:
df.groupby(["key1","key2"]).mean()

data1     data2
key1 key2                    
a    1    -0.264609 -1.679621
     2    -1.465616 -0.379823
b    1     0.171047 -0.518360
     2     0.910231  0.362820

In [21]:
# regardless of the objective in using .groupby(), generally useful GroupBy method is .size()
# .size() returns a Series containing group sizes:

df.groupby(["key1","key2"]).size()

key1  key2
a     1       1
      2       1
b     1       1
      2       1
dtype: int64

In [22]:
# any missing values in a group key are excluded from the result by default
# this behavior can be disabled by passing dropna=False to .groupby:

df.groupby("key1",dropna=False).size()

key1
a      3
b      2
NaN    2
dtype: int64

In [23]:
df.groupby(["key1","key2"],dropna=False).size()

key1  key2
a     1       1
      2       1
      <NA>    1
b     1       1
      2       1
NaN   1       2
dtype: int64

In [24]:
# a group function similar to .size() is .count(), which computes the number of non-null values in each group:

df.groupby("key1").count()

,key2,data1,data2
key1,,,
a,2,3,3
b,2,2,2


In [30]:
# Iterating over Groups
# the object returned by .groupby() supports iteration, generating a sequence of 2-tuples containing the group name along with the chunk of data

df

,key1,key2,data1,data2
0,a,1,-0.264609,-1.679621
1,a,2,-1.465616,-0.379823
2,None,1,-0.927017,0.051805
3,b,2,0.910231,0.362820
4,b,1,0.171047,-0.518360
5,a,<NA>,-0.922581,-0.976857
6,None,1,0.699284,-1.292389


In [31]:
for fuck,shit in df.groupby("key1"):
    print(fuck) # print the label, which is indexes under key1
    print(shit) # print the corresponding data to the label

# for loop above uses 2 variables - 1 for label and 1 for the corresponding data
# .groupby() requires 2 variables in for loop, since .groupby yields a tuple containing (Label, Data)

a
  key1  key2     data1     data2
0    a     1 -0.264609 -1.679621
1    a     2 -1.465616 -0.379823
5    a  <NA> -0.922581 -0.976857
b
  key1  key2     data1    data2
3    b     2  0.910231  0.36282
4    b     1  0.171047 -0.51836


In [33]:
# in the case of multiple keys, the first elemnt in the tuple will be a tuple of key values:
for(k1,k2),group in df.groupby(["key1","key2"]):
    print((k1,k2))
    print(group)

# for loop above : (k1,k2) and group are variables

('a', np.int64(1))
  key1  key2     data1     data2
0    a     1 -0.264609 -1.679621
('a', np.int64(2))
  key1  key2     data1     data2
1    a     2 -1.465616 -0.379823
('b', np.int64(1))
  key1  key2     data1    data2
4    b     1  0.171047 -0.51836
('b', np.int64(2))
  key1  key2     data1    data2
3    b     2  0.910231  0.36282


In [36]:
# you can choose to do whatever you want with the pieces of data
# computing a dictionary of the data pieces as a one-liner:
pieces = {name:group for name,group in df.groupby("key1")}
pieces["b"]

,key1,key2,data1,data2
3,b,2,0.910231,0.36282
4,b,1,0.171047,-0.51836


In [38]:
# by default .groupby() groups on axis="index" but you can group on any other of the axes
# grouping the columns of the df above by whether they start with "key" or "data":

grouped = df.groupby({"key1":"key","key2":"key","data1":"data","data2":"data"},axis="columns")
grouped

C:\Users\Public\Documents\ESTsoft\CreatorTemp\ipykernel_21200\3589335613.py:4: FutureWarning: DataFrame.groupby with axis=1 is deprecated. Do `frame.T.groupby(...)` without axis instead.
  grouped = df.groupby({"key1":"key","key2":"key","data1":"data","data2":"data"},axis="columns")


In [39]:
# we can print out the above groups like so:

for group_key,group_values in grouped:
    print(group_key)
    print(group_values)

data
      data1     data2
0 -0.264609 -1.679621
1 -1.465616 -0.379823
2 -0.927017  0.051805
3  0.910231  0.362820
4  0.171047 -0.518360
5 -0.922581 -0.976857
6  0.699284 -1.292389
key
   key1  key2
0     a     1
1     a     2
2  None     1
3     b     2
4     b     1
5     a  <NA>
6  None     1


In [40]:
# Grouping with Dictionaries and Series

# grouping information may exist in a form other than an array

people = pd.DataFrame(np.random.standard_normal((5,5)),
                      columns=["a","b","c","d","e"],
                      index=["Joe","Steve","Wanda","Jill","Trey"])
people

,a,b,c,d,e
Joe,0.721969,0.166370,1.082801,-0.700231,0.468084
Steve,-0.519197,-0.739146,-0.222925,1.025699,0.645954
Wanda,1.216494,-0.561401,-0.129135,-0.312661,0.977702
Jill,0.666911,-0.475942,-0.402706,-2.235065,-0.703799
Trey,0.109867,-1.550332,0.546729,-0.282224,-1.252048


In [41]:
people.iloc[2:3,[1,2]]=np.nan # adding a few NaN values
people

,a,b,c,d,e
Joe,0.721969,0.166370,1.082801,-0.700231,0.468084
Steve,-0.519197,-0.739146,-0.222925,1.025699,0.645954
Wanda,1.216494,NaN,NaN,-0.312661,0.977702
Jill,0.666911,-0.475942,-0.402706,-2.235065,-0.703799
Trey,0.109867,-1.550332,0.546729,-0.282224,-1.252048


In [42]:
# suppose we get a group correspondence for the coluns and want to sum the columns by group:

mapping = {"a":"red","b":"red","c":"blue","d":"blue","e":"red","f":"orange"}
mapping

{'a': 'red', 'b': 'red', 'c': 'blue', 'd': 'blue', 'e': 'red', 'f': 'orange'}

In [44]:
# now, construct an array from this dictionary to pass to a .groupby()
# but instead we can just pass the dictionary (additional key "f" has been added to show unused grouping keys are okay)

by_column = people.groupby(mapping,axis="columns")
by_column

C:\Users\Public\Documents\ESTsoft\CreatorTemp\ipykernel_21200\94912838.py:4: FutureWarning: DataFrame.groupby with axis=1 is deprecated. Do `frame.T.groupby(...)` without axis instead.
  by_column = people.groupby(mapping,axis="columns")


In [46]:
by_column.sum()

# this is like conditional sum. By assiging grouping logic, given in dictionary, data can be grouped and aggregated.

,blue,red
Joe,0.382570,1.356423
Steve,0.802773,-0.612389
Wanda,-0.312661,2.194195
Jill,-2.637772,-0.512830
Trey,0.264505,-2.692512


In [47]:
# the same functionality holds for Series, which can be viewed as a fixed-size mapping:

map_series = pd.Series(mapping)
map_series

a       red
b       red
c      blue
d      blue
e       red
f    orange
dtype: object

In [50]:
people.groupby(map_series,axis="columns").count()

C:\Users\Public\Documents\ESTsoft\CreatorTemp\ipykernel_21200\172532021.py:1: FutureWarning: DataFrame.groupby with axis=1 is deprecated. Do `frame.T.groupby(...)` without axis instead.
  people.groupby(map_series,axis="columns").count()


,blue,red
Joe,2,3
Steve,2,3
Wanda,1,2
Jill,2,3
Trey,2,3
